# 24 — SDP Campaign Performance Validation v1

This notebook adds the missing validation layer: where has the SDP actually stood, and how do those wards compare to the model?

It can work in two modes:

1. **Auto-extract mode**: filter SDP candidate rows from `local_election_results_raw_v1.csv` or equivalent candidate-level election file.
2. **Manual SDP file mode**: load `sdp_candidate_results_v1.csv`, plus an optional `sdp_candidate_results_2026_raw_v1.csv`.

For 2026, the recommended approach is a provisional manual file. Use `boundary_year`, `ward_code`, and an optional manual mapping field `WD25CD` where the 2026 ward is known to correspond to the current atlas ward. Rows that cannot be mapped are still retained for review.

In [1]:
from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
DICTIONARY_DIR = DATA_DIR / "dictionaries"

for d in [PROCESSED_DIR, GEOGRAPHY_DIR, DICTIONARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Processed:", PROCESSED_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Processed: c:\Users\keena\Documents\Electoral_Tribes\data\processed


In [2]:
OUTPUT_DIR = PROCESSED_DIR / "sdp_campaign_validation_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# User-provided expected counts. Used only as a QA reference, not as source data.
EXPECTED_SDP_CANDIDATE_COUNTS = {
    2021: 68,
    2022: 30,
    2023: 36,
    2024: 28,
    2025: 11,
    2026: 48,
}

print("Output:", OUTPUT_DIR)

Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1


In [3]:
def find_file(filename, search_dirs=None, required=True):
    if search_dirs is None:
        search_dirs = [
            PROCESSED_DIR / "target_review_pack_v1",
            PROCESSED_DIR / "target_model_v2",
            PROCESSED_DIR / "target_model_v1",
            PROCESSED_DIR / "report_assets_v1",
            PROCESSED_DIR / "election_results",
            PROCESSED_DIR,
            DATA_DIR / "raw" / "election_results",
            DATA_DIR / "raw",
            PROJECT_DIR,
            Path.cwd(),
        ]
    for folder in search_dirs:
        path = folder / filename
        if path.exists():
            return path
    # recursive fallback under processed and data/raw
    for root in [PROCESSED_DIR, DATA_DIR / "raw"]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(f"Could not find {filename}")
    return None


def read_csv(filename, required=True):
    path = find_file(filename, required=required)
    if path is None:
        print("Optional file missing:", filename)
        return None, None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {path}")
    return df, path


def save_csv(df, path, index=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print("Saved:", path, df.shape)
    return path


def boolish(s):
    return s.fillna(False).astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])


def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def safe_col(df, col, default=np.nan):
    if col in df.columns:
        return df[col]
    return pd.Series(default, index=df.index)


def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = x.replace("&", " and ")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

## 24.1 Required / optional inputs

Preferred manual schema for `sdp_candidate_results_v1.csv` and `sdp_candidate_results_2026_raw_v1.csv`:

```text
election_year
election_date
council_name
lad_code
ward_name
ward_code
boundary_year
candidate_name
party_label
sdp_votes
sdp_vote_share
sdp_position
winner_party
runner_up_party
valid_votes
turnout
source_url
source_notes
WD25CD        # optional, recommended if already mapped
LAD25CD       # optional
```

If this file is absent, the notebook tries to extract SDP rows from the raw candidate-level election database.

In [4]:
# Write a template for manual SDP data entry.
template_cols = [
    "election_year", "election_date", "council_name", "lad_code", "ward_name", "ward_code", "boundary_year",
    "candidate_name", "party_label", "sdp_votes", "sdp_vote_share", "sdp_position",
    "winner_party", "runner_up_party", "valid_votes", "turnout", "source_url", "source_notes", "WD25CD", "LAD25CD"
]
template = pd.DataFrame(columns=template_cols)
save_csv(template, OUTPUT_DIR / "sdp_candidate_results_template_v1.csv")

expected_counts = pd.DataFrame([{"election_year": y, "expected_sdp_candidates_user_provided": c} for y, c in EXPECTED_SDP_CANDIDATE_COUNTS.items()])
save_csv(expected_counts, OUTPUT_DIR / "sdp_expected_candidate_counts_user_provided_v1.csv")

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_candidate_results_template_v1.csv (0, 20)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_expected_candidate_counts_user_provided_v1.csv (6, 2)


WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/sdp_campaign_validation_v1/sdp_expected_candidate_counts_user_provided_v1.csv')

## 24.2 Load SDP rows

The notebook first loads manual SDP files if available. If not, it extracts from the candidate-level election data.

In [5]:
def is_sdp_party_label(s):
    s = str(s).lower().strip()
    if not s or s == "nan":
        return False
    # Avoid SDLP false positives.
    if "sdlp" in s:
        return False
    return (s == "sdp") or ("social democratic party" in s) or re.search(r"\bsdp\b", s) is not None


def standardise_sdp_columns(df, source_label="unknown"):
    df = df.copy()
    # Flexible renames.
    renames = {
        "votes": "sdp_votes",
        "position": "sdp_position",
        "party": "party_label",
        "raw_party_label": "party_label",
        "ward_code": "ward_code",
        "lad_code": "lad_code",
    }
    for old, new in renames.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    # Ensure required columns exist.
    for col in template_cols:
        if col not in df.columns:
            df[col] = np.nan

    df["sdp_votes"] = to_num(df["sdp_votes"])
    df["valid_votes"] = to_num(df["valid_votes"])
    df["sdp_vote_share"] = to_num(df["sdp_vote_share"])
    # If vote share missing, compute where possible.
    mask = df["sdp_vote_share"].isna() & df["sdp_votes"].notna() & df["valid_votes"].gt(0)
    df.loc[mask, "sdp_vote_share"] = df.loc[mask, "sdp_votes"] / df.loc[mask, "valid_votes"]
    df["source_mode"] = source_label
    return df

manual, _ = read_csv("sdp_candidate_results_v1.csv", required=False)
manual_2026, _ = read_csv("sdp_candidate_results_2026_raw_v1.csv", required=False)

sdp_frames = []
if manual is not None:
    sdp_frames.append(standardise_sdp_columns(manual, "manual_sdp_candidate_results_v1"))
if manual_2026 is not None:
    sdp_frames.append(standardise_sdp_columns(manual_2026, "manual_2026_sdp_results"))

# Auto-extract from candidate-level election database if available.
raw_candidates, _ = read_csv("local_election_results_raw_v1.csv", required=False)
if raw_candidates is None:
    raw_candidates, _ = read_csv("local_election_results_candidates_raw_v1.csv", required=False)
if raw_candidates is None:
    raw_candidates, _ = read_csv("candidates_all_normalised_v1.csv", required=False)

if raw_candidates is not None:
    party_col = None
    for c in ["party_label", "raw_party_label", "party"]:
        if c in raw_candidates.columns:
            party_col = c
            break
    if party_col:
        extracted = raw_candidates[raw_candidates[party_col].apply(is_sdp_party_label)].copy()
        if len(extracted):
            sdp_frames.append(standardise_sdp_columns(extracted, "auto_extracted_candidate_database"))
            print("Auto-extracted SDP rows:", len(extracted))

if not sdp_frames:
    print("No SDP rows found yet. Fill sdp_candidate_results_template_v1.csv and rerun this notebook.")
    sdp = pd.DataFrame(columns=template_cols + ["source_mode"])
else:
    sdp = pd.concat(sdp_frames, ignore_index=True, sort=False)
    # Remove duplicate exact candidate rows where manual and auto-extract overlap.
    dedupe_cols = [c for c in ["election_year", "council_name", "ward_name", "candidate_name", "sdp_votes"] if c in sdp.columns]
    sdp = sdp.drop_duplicates(dedupe_cols)

print("SDP rows loaded:", len(sdp))
display(sdp.head())

Optional file missing: sdp_candidate_results_v1.csv
Optional file missing: sdp_candidate_results_2026_raw_v1.csv
Loaded local_election_results_raw_v1.csv: (80392, 52) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\local_election_results_raw_v1.csv
Auto-extracted SDP rows: 171
SDP rows loaded: 171


,result_id,result_area_key,source_year,election_date,election_year,election_type,ordinary_or_by_election,source_file,source_sheet,county_name,council_name,lad_code,upper_tier_authority,lower_tier_authority,ward_name,standard_ward_name,ward_code,ec_ward_code,boundary_year,geography_type,atlas_join_strategy,seats_available,electorate,turnout,valid_votes,ballots,invalid_votes,candidate_number,candidate_name,candidate_first_names,candidate_last_names,candidate_gender,incumbent,party_label,standard_party_label,party_family,party_group,party_id,sdp_votes,vote_share,elected,rank,votes_effective,matched_wd25cd,matched_wd25nm,matched_lad25cd,matched_lad25nm,source_url,source_notes,data_quality_flag,manual_review_required,review_status,sdp_vote_share,sdp_position,winner_party,runner_up_party,WD25CD,LAD25CD,source_mode
0,717c6d0118395f8b,2021|CODE|BRISTOL_CITY_OF|E05010899|FROME_VALE,2021,2021-05-06,2021,NaN,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",E06000023,NaN,"Bristol, City Of",Frome Vale,Frome Vale,E05010899,NaN,2021,electoral_ward_or_division,needs_historical_ward_crosswalk,NaN,NaN,NaN,3835.0,NaN,NaN,9.0,Trueman T.,NaN,NaN,M,False,SDP,SDP,SDP,OTH,403.0,112,0.029205,False,NaN,True,NaN,NaN,NaN,NaN,NaN,House of Commons Library Local Election Handbo...,ok,False,suggested,0.029205,NaN,NaN,NaN,NaN,NaN,auto_extracted_candidate_database
1,eba25911b2760b0d,2021|CODE|READING|E05002321|CAVERSHAM,2021,2021-05-06,2021,NaN,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,Reading,E06000038,NaN,Reading,Caversham,Caversham,E05002321,NaN,2021,electoral_ward_or_division,needs_historical_ward_crosswalk,NaN,NaN,NaN,3112.0,NaN,NaN,6.0,Skelton D.J.A.,NaN,NaN,M,False,SDP,SDP,SDP,OTH,403.0,17,0.005463,False,NaN,True,NaN,NaN,NaN,NaN,NaN,House of Commons Library Local Election Handbo...,ok,False,suggested,0.005463,NaN,NaN,NaN,NaN,NaN,auto_extracted_candidate_database
2,1c032f9d83a00db2,2021|CODE|BUCKINGHAMSHIRE|E05013159|STONE_AND_...,2021,2021-05-06,2021,NaN,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,Buckinghamshire,E06000060,NaN,Buckinghamshire,Stone And Waddesdon,Stone And Waddesdon,E05013159,NaN,2021,electoral_ward_or_division,needs_historical_ward_crosswalk,NaN,NaN,NaN,4005.0,NaN,NaN,15.0,Tinay P.D.,NaN,NaN,M,False,SDP,SDP,SDP,OTH,406.0,37,0.009238,False,NaN,True,NaN,NaN,NaN,NaN,NaN,House of Commons Library Local Election Handbo...,ok,False,suggested,0.009238,NaN,NaN,NaN,NaN,NaN,auto_extracted_candidate_database
3,e7e49d776923a05c,2021|CODE|HARTLEPOOL|E05013038|BURN_VALLEY,2021,2021-05-06,2021,NaN,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,Hartlepool,E06000001,NaN,Hartlepool,Burn Valley,Burn Valley,E05013038,NaN,2021,electoral_ward_or_division,needs_historical_ward_crosswalk,NaN,NaN,NaN,3218.0,NaN,NaN,7.0,Humphries L.P.,NaN,NaN,F,False,SDP,SDP,SDP,OTH,404.0,225,0.069919,False,NaN,True,NaN,NaN,NaN,NaN,NaN,House of Commons Library Local Election Handbo...,ok,False,suggested,0.069919,NaN,NaN,NaN,NaN,NaN,auto_extracted_candidate_database
4,6630db5908a2d8e8,2021|CODE|WEALDEN|E58000368|HAILSHAM_MARKET,2021,2021-05-06,2021,NaN,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,East Sussex,Wealden,E07000065,NaN,Wealden,Hailsham Market,Hailsham Market,E58000368,NaN,2021,county_electoral_division,needs_county_electoral_division_geography,NaN,NaN,NaN,2484.0,NaN,NaN,3.0,Gander S.R.,NaN,NaN,M,False,SDP,SDP,SDP,OTH,401.0,103,0.041465,False,NaN,True,NaN,NaN,NaN,NaN,NaN,House of Commons Library Local Election Handbo...,ok,True,review,0.041465,NaN,NaN,NaN,NaN,NaN,auto_extracted_candidate_database


## 24.3 Candidate count QA

In [6]:
if len(sdp):
    actual_counts = sdp.groupby("election_year", dropna=False).size().reset_index(name="observed_sdp_candidate_rows")
    expected = pd.DataFrame([{"election_year": y, "expected_user_provided": c} for y, c in EXPECTED_SDP_CANDIDATE_COUNTS.items()])
    count_check = expected.merge(actual_counts, on="election_year", how="outer").sort_values("election_year")
    count_check["difference_observed_minus_expected"] = count_check["observed_sdp_candidate_rows"].fillna(0) - count_check["expected_user_provided"].fillna(0)
else:
    count_check = pd.DataFrame(columns=["election_year", "expected_user_provided", "observed_sdp_candidate_rows", "difference_observed_minus_expected"])

save_csv(count_check, OUTPUT_DIR / "sdp_candidate_count_check_v1.csv")
display(count_check)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_candidate_count_check_v1.csv (6, 4)


,election_year,expected_user_provided,observed_sdp_candidate_rows,difference_observed_minus_expected
0,2021,68,68.0,0.0
1,2022,30,28.0,-2.0
2,2023,36,36.0,0.0
3,2024,28,28.0,0.0
4,2025,11,11.0,0.0
5,2026,48,NaN,-48.0


## 24.4 Join SDP rows to the model atlas/review file

Matching priority:

1. `WD25CD` if supplied;
2. exact `ward_code` to `WD25CD` if boundary is compatible;
3. name/council fallback for review only.

Rows that cannot be mapped are retained in `sdp_unmatched_results_review_v1.csv`.

In [7]:
review, _ = read_csv("all_available_revised_consolidated_review_v1.csv", required=False)
if review is None:
    review, _ = read_csv("all_available_consolidated_target_review_v1.csv", required=False)
if review is None:
    review, _ = read_csv("north_west_revised_consolidated_review_v1.csv", required=False)
if review is None:
    review, _ = read_csv("north_west_consolidated_target_review_v1.csv", required=False)

if review is None:
    print("No model review file found. SDP rows will be standardised but not profiled against tribes.")
else:
    print("Review table:", review.shape)

# Manual 2026 mapping file, if needed.
manual_map, _ = read_csv("sdp_2026_ward_mapping_manual_v1.csv", required=False)
if manual_map is not None:
    # Expected columns: election_year, council_name, ward_name, ward_code, WD25CD, LAD25CD, mapping_notes
    join_cols = [c for c in ["election_year", "council_name", "ward_name", "ward_code"] if c in sdp.columns and c in manual_map.columns]
    if join_cols:
        sdp = sdp.merge(
            manual_map[[*join_cols, "WD25CD", "LAD25CD", "mapping_notes"]].drop_duplicates(join_cols),
            on=join_cols,
            how="left",
            suffixes=("", "_manualmap")
        )
        if "WD25CD_manualmap" in sdp.columns:
            sdp["WD25CD"] = sdp["WD25CD"].combine_first(sdp["WD25CD_manualmap"])
        if "LAD25CD_manualmap" in sdp.columns:
            sdp["LAD25CD"] = sdp["LAD25CD"].combine_first(sdp["LAD25CD_manualmap"])

# If no WD25CD but ward_code is already a WD25 code, use it.
if "WD25CD" in sdp.columns and "ward_code" in sdp.columns:
    sdp["WD25CD"] = sdp["WD25CD"].combine_first(sdp["ward_code"])

if review is not None and len(sdp):
    profile_cols = [c for c in [
        "WD25CD", "WD25NM", "LAD25CD", "LAD25NM", "analysis_region", "revised_strategic_lane", "strategic_lane", "report_confidence_band",
        "initial_watchlist_score", "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score",
        "dominant_cluster_name", "second_cluster_name", "latest_election_top_party_bucket", "latest_election_runner_up_party_bucket",
        "conservative_transition_score", "labour_stronghold_breakthrough_score", "reform_independent_disruption_score", "green_ld_noncore_score", "primary_party_transition_diagnostic"
    ] if c in review.columns]
    sdp_profile = sdp.merge(review[profile_cols].drop_duplicates("WD25CD"), on="WD25CD", how="left", suffixes=("", "_model"))
else:
    sdp_profile = sdp.copy()

sdp_profile["matched_to_model"] = sdp_profile.get("dominant_cluster_name", pd.Series(np.nan, index=sdp_profile.index)).notna()
print("Matched to model:", sdp_profile["matched_to_model"].sum(), "of", len(sdp_profile))

Loaded all_available_revised_consolidated_review_v1.csv: (7572, 59) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats\all_available_revised_consolidated_review_v1.csv
Review table: (7572, 59)
Optional file missing: sdp_2026_ward_mapping_manual_v1.csv
Matched to model: 107 of 171


## 24.5 Save SDP validation outputs

In [8]:
save_csv(sdp, OUTPUT_DIR / "sdp_candidate_results_standardised_v1.csv")
save_csv(sdp_profile, OUTPUT_DIR / "sdp_campaign_wards_profile_v1.csv")

unmatched = sdp_profile[~sdp_profile["matched_to_model"]].copy() if len(sdp_profile) else sdp_profile.copy()
save_csv(unmatched, OUTPUT_DIR / "sdp_unmatched_results_review_v1.csv")

if len(sdp_profile):
    # Highest vote share cases.
    save_csv(
        sdp_profile.sort_values("sdp_vote_share", ascending=False),
        OUTPUT_DIR / "sdp_highest_vote_share_cases_v1.csv"
    )

    # Repeat campaign wards.
    repeat = sdp_profile.groupby(["WD25CD", "ward_name", "council_name"], dropna=False).agg(
        sdp_contests=("election_year", "nunique"),
        first_year=("election_year", "min"),
        latest_year=("election_year", "max"),
        max_sdp_vote_share=("sdp_vote_share", "max"),
        mean_sdp_vote_share=("sdp_vote_share", "mean"),
        total_sdp_votes=("sdp_votes", "sum"),
    ).reset_index().sort_values(["sdp_contests", "max_sdp_vote_share"], ascending=[False, False])
    save_csv(repeat, OUTPUT_DIR / "sdp_repeat_campaign_wards_v1.csv")

    # Group summaries.
    for group_col, filename in [
        ("dominant_cluster_name", "sdp_performance_by_dominant_tribe_v1.csv"),
        ("second_cluster_name", "sdp_performance_by_second_tribe_v1.csv"),
        ("latest_election_top_party_bucket", "sdp_performance_by_latest_top_party_v1.csv"),
        ("analysis_region", "sdp_performance_by_region_v1.csv"),
        ("revised_strategic_lane", "sdp_performance_by_strategic_lane_v1.csv"),
        ("primary_party_transition_diagnostic", "sdp_performance_by_transition_diagnostic_v1.csv"),
    ]:
        if group_col in sdp_profile.columns:
            summary = sdp_profile.groupby(group_col, dropna=False).agg(
                sdp_candidate_rows=("candidate_name", "count"),
                contests=("election_year", "nunique"),
                mean_sdp_vote_share=("sdp_vote_share", "mean"),
                median_sdp_vote_share=("sdp_vote_share", "median"),
                max_sdp_vote_share=("sdp_vote_share", "max"),
                total_sdp_votes=("sdp_votes", "sum"),
                mean_model_score=("initial_watchlist_score", "mean") if "initial_watchlist_score" in sdp_profile.columns else ("sdp_votes", "sum"),
            ).reset_index().sort_values("mean_sdp_vote_share", ascending=False)
            save_csv(summary, OUTPUT_DIR / filename)

    # Model score vs SDP vote share.
    score_cols = [c for c in ["initial_watchlist_score", "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score", "sdp_vote_share"] if c in sdp_profile.columns]
    if len(score_cols) >= 2:
        corr = sdp_profile[score_cols].corr(numeric_only=True)
        save_csv(corr.reset_index(), OUTPUT_DIR / "sdp_campaign_vs_model_score_correlation_v1.csv")

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_candidate_results_standardised_v1.csv (171, 59)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_campaign_wards_profile_v1.csv (171, 76)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_unmatched_results_review_v1.csv (64, 76)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_highest_vote_share_cases_v1.csv (171, 76)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_repeat_campaign_wards_v1.csv (148, 9)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_performance_by_dominant_tribe_v1.csv (8, 8)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_performance_by_second_tribe_v1.csv (7, 8)
Saved: c:\Users\keena\Documents\Electora

## 24.6 2026 handling note

For 2026, the best immediate approach is a provisional manual SDP-only file with `boundary_year = 2026`. Add `WD25CD` where you are confident the 2026 ward maps to the atlas ward. Keep uncertain rows unmatched rather than forcing a false join.

When a formal 2026 ward lookup/boundary file exists, rerun the geography crosswalk pipeline and then rerun this notebook.